In [1]:
import torch
import torch.nn.functional as F

# Import your project modules
import config as cfg
import cnn_utils.data as data_utils
from cnn_utils.model import load_model
from cnn_utils.evaluate import EvalClassification

# 1. Load the Phase 6 Linear Baseline Model
# (Change to a Phase 7 checkpoint if your Gated Attention numbers were from Phase 7)
MODEL_PATH = cfg.ph6_linear_best 
print(f"Loading model from: {MODEL_PATH}")
model_linear = load_model(MODEL_PATH)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_linear.to(device)
model_linear.eval()

# 2. Load the Dataloaders
print("Loading Lab Dataloaders...")
_, _, test_dl_lab = data_utils.get_3_dataloaders(cfg)

print("Loading OOD Dataloaders...")
_, test_dl_ood = data_utils.get_ood_dataloaders(cfg)

# 3. Initialize Evaluators
eval_lab = EvalClassification(cfg, model_linear, test_dl_lab)
eval_ood = EvalClassification(cfg, model_linear, test_dl_ood)

/home/takayuki/.local/lib/python3.10/site-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(
/home/takayuki/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


Loading model from: /home/takayuki/Desktop/summer2025/plants/training/phase6/models/best_model_07-14_14-56--10_p6_linear_convnextv2_tiny.fcmae_ft_in22k_in1k.pth
Model loaded successfully from /home/takayuki/Desktop/summer2025/plants/training/phase6/models/best_model_07-14_14-56--10_p6_linear_convnextv2_tiny.fcmae_ft_in22k_in1k.pth
Loading Lab Dataloaders...
 Total dataset size 	: 230
 Train dataset size 	: 138
 Val dataset size 	: 46
 Test dataset size 	: 46

Loading OOD Dataloaders...
Found 10 classes locally: ['Hydrocharis morsus-ranae', 'Myriophyllum spicatum', 'Nitellopsis obtusa', 'Nuphar variegata', 'Potamogeton crispus', 'Potamogeton gramineus', 'Potamogeton illinoensis', 'Potamogeton richardsonii', 'Potamogeton robbinsii', 'Ranunculus aquatilis']

OOD split complete:
  OOD Validation set size: 16
  OOD Test set size    : 17



In [2]:
# Cell 2
# 1. Load the Phase 7 Gated Attention Model
GATED_MODEL_PATH = cfg.ph7_gated_attn_best_ood_acc 
print(f"Loading Gated Attention model from: {GATED_MODEL_PATH}")

model_gated = load_model(GATED_MODEL_PATH)
model_gated.to(device)
model_gated.eval()

# 2. Initialize Evaluators for Gated Attention
eval_lab_gated = EvalClassification(cfg, model_gated, test_dl_lab)
eval_ood_gated = EvalClassification(cfg, model_gated, test_dl_ood)

Loading Gated Attention model from: /home/takayuki/Desktop/summer2025/plants/training/phase7/models/best_ood_acc_model_07-31_04-19--55_ph7_gated_attn_actual_convnextv2_tiny.fcmae_ft_in22k_in1k.pth
Model loaded successfully from /home/takayuki/Desktop/summer2025/plants/training/phase7/models/best_ood_acc_model_07-31_04-19--55_ph7_gated_attn_actual_convnextv2_tiny.fcmae_ft_in22k_in1k.pth


In [3]:
# Cell 3
# Dynamically override the configuration variables for patching and YNLT

# 1. Patching Parameters
cfg.patching_method = 'multi-scale'
cfg.padding_logic = 'reflect'

cfg.ms_scale = [0.2, 0.3, 0.4, 0.5] 
cfg.ms_overlap = [0.0, 0.1, 0.2, 0.3] 

# 2. Confusion Subsystem Parameters
cfg.do_critical_review = True
cfg.critical_confusion_level = 1       
cfg.margin_threshold = 0.2
cfg.entropy_threshold = 2.0
cfg.confidence_threshold = 0.5
cfg.ignore_invasive_predictions = True 

print("YNLT and Patching Hyperparameters configured:")
print(f"Method: {cfg.patching_method}")
print(f"Scales: {cfg.ms_scale}")
print(f"Overlaps: {cfg.ms_overlap}")

YNLT and Patching Hyperparameters configured:
Method: multi-scale
Scales: [0.2, 0.3, 0.4, 0.5]
Overlaps: [0.0, 0.1, 0.2, 0.3]


In [4]:
# Cell 4
from cnn_utils.predict import PredictPlants 

# Initialize the YNLT Predictors
predictor_linear = PredictPlants(model_linear, "Linear Baseline")
predictor_gated = PredictPlants(model_gated, "Gated Attention")

# Create wrapper functions to match the signature expected by evaluator.evaluate()
# EvalClassification passes (images, label_item) to the predict_function
def predict_fn_linear_ynlt(images, label):
    return predictor_linear.predict(input=images, true_label=label, return_fig=False, display=False)

def predict_fn_gated_ynlt(images, label):
    return predictor_gated.predict(input=images, true_label=label, return_fig=False, display=False)

/home/takayuki/.local/lib/python3.10/site-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


In [5]:
# Cell 5: Comprehensive LAB Ablation Table
import pandas as pd

print("Running Comprehensive Ablation on LAB Test Set...\n")

# Helper function to extract all metrics cleanly
def get_metrics_lab(evaluator, predict_fn=None):
    if predict_fn:
        evaluator.evaluate(predict_function=predict_fn)
    else:
        evaluator.evaluate()
        
    class_acc = evaluator.get_accuracy(verbose=False)
    bin_metrics = evaluator.get_binary_metrics(display=False)
    
    return {
        "Class Acc (%)": class_acc * 100,
        "Bin Acc (%)": bin_metrics['Accuracy'] * 100,
        "Bin FNR (%)": bin_metrics['FNR'] * 100
    }

# 1. Linear Classifier (Base)
print("Evaluating: Linear Classifier")
lin_lab_metrics = get_metrics_lab(eval_lab)

# 2. Linear + YNLT
print("Evaluating: Linear Classifier + YNLT")
lin_ynlt_lab_metrics = get_metrics_lab(eval_lab, predict_fn_linear_ynlt)

# 3. Gated Attention (Base)
print("Evaluating: Gated Attention")
gat_lab_metrics = get_metrics_lab(eval_lab_gated)

# 4. Gated Attention + YNLT
print("Evaluating: Gated Attention + YNLT")
gat_ynlt_lab_metrics = get_metrics_lab(eval_lab_gated, predict_fn_gated_ynlt)

# Compile Lab Results into a clean DataFrame
lab_results = pd.DataFrame([
    {"Model": "Linear Classifier", **lin_lab_metrics},
    {"Model": "Linear + YNLT", **lin_ynlt_lab_metrics},
    {"Model": "Gated Attention", **gat_lab_metrics},
    {"Model": "Gated Attention + YNLT", **gat_ynlt_lab_metrics}
])

print("\n--- Lab Data Results ---")
print(lab_results.round(2).to_string(index=False))

Running Comprehensive Ablation on LAB Test Set...

Evaluating: Linear Classifier


  -- Evaluating: 100%|██████████| 46/46 [00:04<00:00,  9.81it/s]


Evaluating: Linear Classifier + YNLT


  -- Evaluating: 100%|██████████| 46/46 [00:22<00:00,  2.07it/s]


Evaluating: Gated Attention


  -- Evaluating: 100%|██████████| 46/46 [00:05<00:00,  8.08it/s]


Evaluating: Gated Attention + YNLT


  -- Evaluating: 100%|██████████| 46/46 [00:18<00:00,  2.54it/s]


--- Lab Data Results ---
                 Model  Class Acc (%)  Bin Acc (%)  Bin FNR (%)
     Linear Classifier          84.78        93.48         30.0
         Linear + YNLT          91.30        93.48         30.0
       Gated Attention          86.96        97.83         10.0
Gated Attention + YNLT          93.48        97.83         10.0


In [6]:
# Cell 6: Comprehensive OOD Ablation Table
print("Running Comprehensive Ablation on OOD Test Set...\n")

# Helper function to extract all metrics cleanly
def get_metrics_ood(evaluator, predict_fn=None):
    if predict_fn:
        evaluator.evaluate(predict_function=predict_fn)
    else:
        evaluator.evaluate()
        
    class_acc = evaluator.get_accuracy(verbose=False)
    bin_metrics = evaluator.get_binary_metrics(display=False)
    
    return {
        "Class Acc (%)": class_acc * 100,
        "Bin Acc (%)": bin_metrics['Accuracy'] * 100,
        "Bin FNR (%)": bin_metrics['FNR'] * 100
    }

# 1. Linear Classifier (Base)
print("Evaluating: Linear Classifier")
lin_metrics = get_metrics_ood(eval_ood)

# 2. Linear + YNLT
print("Evaluating: Linear Classifier + YNLT")
lin_ynlt_metrics = get_metrics_ood(eval_ood, predict_fn_linear_ynlt)

# 3. Gated Attention (Base)
print("Evaluating: Gated Attention")
gat_metrics = get_metrics_ood(eval_ood_gated)

# 4. Gated Attention + YNLT
print("Evaluating: Gated Attention + YNLT")
gat_ynlt_metrics = get_metrics_ood(eval_ood_gated, predict_fn_gated_ynlt)

# Compile Results into a clean DataFrame
ood_results = pd.DataFrame([
    {"Model": "Linear Classifier", **lin_metrics},
    {"Model": "Linear + YNLT", **lin_ynlt_metrics},
    {"Model": "Gated Attention", **gat_metrics},
    {"Model": "Gated Attention + YNLT", **gat_ynlt_metrics}
])

print("\n--- Out-Of-Distribution (OOD) Data Results ---")
print(ood_results.round(2).to_string(index=False))

Running Comprehensive Ablation on OOD Test Set...

Evaluating: Linear Classifier


  -- Evaluating: 100%|██████████| 17/17 [00:01<00:00,  9.24it/s]


Evaluating: Linear Classifier + YNLT


  -- Evaluating: 100%|██████████| 17/17 [00:21<00:00,  1.26s/it]


Evaluating: Gated Attention


  -- Evaluating: 100%|██████████| 17/17 [00:02<00:00,  8.29it/s]


Evaluating: Gated Attention + YNLT


  -- Evaluating: 100%|██████████| 17/17 [00:10<00:00,  1.61it/s]


--- Out-Of-Distribution (OOD) Data Results ---
                 Model  Class Acc (%)  Bin Acc (%)  Bin FNR (%)
     Linear Classifier          47.06        58.82         62.5
         Linear + YNLT          52.94        58.82         62.5
       Gated Attention          52.94        76.47         12.5
Gated Attention + YNLT          64.71        76.47         12.5


In [7]:
# Quick diagnostic to see if YNLT is triggering
trigger_count = 0
total_images = len(test_dl_ood.dataset)

for images, labels in test_dl_ood:
    images = images.to(device)
    # Get the raw prediction dictionary (with display=False to avoid spamming the output)
    pred_dict = predictor_gated.predict(input=images, true_label=labels.item(), return_fig=False, display=False)
    
    if pred_dict.get("is_critical", False):
        trigger_count += 1

print(f"YNLT triggered on {trigger_count} out of {total_images} OOD images.")

YNLT triggered on 4 out of 17 OOD images.


In [8]:
# Cell 8: Comprehensive Sanity Check (All Models x All Splits)
import pandas as pd

print("Loading ALL Dataloaders...\n")
# Load Lab Splits
train_dl_lab, val_dl_lab, test_dl_lab_full = data_utils.get_3_dataloaders(cfg)
# Load OOD Splits
val_dl_ood, test_dl_ood_full = data_utils.get_ood_dataloaders(cfg)

# Map out the splits
splits = {
    "Lab: Train": train_dl_lab,
    "Lab: Val": val_dl_lab,
    "Lab: Test": test_dl_lab_full,
    "OOD: Val": val_dl_ood,
    "OOD: Test": test_dl_ood_full
}

# Define the 4 model configurations
model_configs = [
    {"name": "Linear Classifier", "model": model_linear, "predict_fn": None},
    {"name": "Linear + YNLT", "model": model_linear, "predict_fn": predict_fn_linear_ynlt},
    {"name": "Gated Attention", "model": model_gated, "predict_fn": None},
    {"name": "Gated Attention + YNLT", "model": model_gated, "predict_fn": predict_fn_gated_ynlt}
]

results_data = []

for config in model_configs:
    print(f"\n--- Evaluating Model: {config['name']} ---")
    
    for split_name, dl in splits.items():
        if dl is None: 
            continue
            
        # Initialize evaluator for this specific model and dataloader
        evaluator = EvalClassification(cfg, config['model'], dl)
        
        print(f"  Running on {split_name}...")
        
        # Run prediction
        if config['predict_fn']:
            evaluator.evaluate(predict_function=config['predict_fn'])
        else:
            evaluator.evaluate()
        
        # Extract metrics
        class_acc = evaluator.get_accuracy(verbose=False)
        bin_metrics = evaluator.get_binary_metrics(display=False)
        
        tp, fn = bin_metrics['TP'], bin_metrics['FN']
        fp, tn = bin_metrics['FP'], bin_metrics['TN']
        total = tp + fn + fp + tn
        
        results_data.append({
            "Model": config['name'],
            "Split": split_name,
            "Total Imgs": total,
            "Class Acc (%)": class_acc * 100,
            "Bin Acc (%)": bin_metrics['Accuracy'] * 100,
            "Bin FNR (%)": bin_metrics['FNR'] * 100,
            "TP (Invasive)": tp,
            "FN (Missed)": fn,
            "FP (Alarm)": fp,
            "TN (Native)": tn
        })

# Compile and display the final table
sanity_check_df = pd.DataFrame(results_data)

print("\n" + "="*100)
print("COMPREHENSIVE SANITY CHECK: ALL MODELS ACROSS ALL SPLITS")
print("="*100)
# Ensure pandas prints all rows without truncating
pd.set_option('display.max_rows', None) 
print(sanity_check_df.round(2).to_string(index=False))

Loading ALL Dataloaders...

 Total dataset size 	: 230
 Train dataset size 	: 138
 Val dataset size 	: 46
 Test dataset size 	: 46

Found 10 classes locally: ['Hydrocharis morsus-ranae', 'Myriophyllum spicatum', 'Nitellopsis obtusa', 'Nuphar variegata', 'Potamogeton crispus', 'Potamogeton gramineus', 'Potamogeton illinoensis', 'Potamogeton richardsonii', 'Potamogeton robbinsii', 'Ranunculus aquatilis']

OOD split complete:
  OOD Validation set size: 16
  OOD Test set size    : 17


--- Evaluating Model: Linear Classifier ---
  Running on Lab: Train...


  -- Evaluating:   0%|          | 0/138 [00:00<?, ?it/s]

  -- Evaluating: 100%|██████████| 138/138 [00:16<00:00,  8.26it/s]


  Running on Lab: Val...


  -- Evaluating: 100%|██████████| 46/46 [00:05<00:00,  8.53it/s]


...

  Running on OOD: Val...


  -- Evaluating: 100%|██████████| 16/16 [00:26<00:00,  1.63s/it]


  Running on OOD: Test...


  -- Evaluating: 100%|██████████| 17/17 [00:22<00:00,  1.30s/it]


COMPREHENSIVE SANITY CHECK: ALL MODELS ACROSS ALL SPLITS
                 Model      Split  Total Imgs  Class Acc (%)  Bin Acc (%)  Bin FNR (%)  TP (Invasive)  FN (Missed)  FP (Alarm)  TN (Native)
     Linear Classifier Lab: Train         138          89.86        97.10        12.50             28            4           0          106
     Linear Classifier   Lab: Val          46          97.83       100.00         0.00              9            0           0           37
     Linear Classifier  Lab: Test          46          84.78        93.48        30.00              7            3           0           36
     Linear Classifier   OOD: Val          16          68.75        87.50        14.29              6            1           1            8
     Linear Classifier  OOD: Test          17          47.06        58.82        62.50              3            5           2            7
         Linear + YNLT Lab: Train         138          82.61        91.30         3.12             31 

In [11]:
# Cell 9 (Updated): Generate Comprehensive PDF Report for Champion Model
import os

# Assuming you saved the provided script as 'utils/validate.py'
from cnn_utils.validate import ReportGenerator

print("Preparing datasets for the PDF report...")

# Fix: Force dataloaders to process 1 image at a time for the manual evaluation loops
test_dl_lab_single = data_utils.make_batch_size_one(test_dl_lab_full)
test_dl_ood_single = data_utils.make_batch_size_one(test_dl_ood_full)

# Package the datasets we want to analyze into a dictionary
datasets_to_test = {
    "Lab Test Data": test_dl_lab_single,
    "OOD Test Data": test_dl_ood_single
}

# Ensure the experiments directory exists (from config)
os.makedirs(cfg.EXPERIMENTS_DIR, exist_ok=True)

print(f"Initializing Report Generator for {predictor_gated.model_name}...")
# Initialize the ReportGenerator using your Gated Attention predictor
report_gen = ReportGenerator(
    predictor=predictor_gated, 
    datasets_to_test=datasets_to_test, 
    experiment_name="Final_Validation_Champion_Model"
)

# Generate the report!
# Pass the single-batch OOD dataloader here
report_gen.generate_pdf_report(dataloader_for_per_sample=test_dl_ood_single)

print(f"\nDone! You can find your PDF report saved in the experiments folder:")
print(report_gen.pdf_path)

Preparing datasets for the PDF report...
Initializing Report Generator for Gated Attention...
Configuration saved as: exp-33_ph7_gated_attn_best_ood_loss_Gated Attention_config.py in /home/takayuki/Desktop/summer2025/plants/plant_classification/../saved_configs
--- Starting Analysis for Model: Gated Attention ---

  - On dataset: Lab Test Data

  - Running baseline evaluation for Gated Attention...


  -- Evaluating: 100%|██████████| 46/46 [00:04<00:00, 10.46it/s]



  - Running critical review evaluation for Gated Attention...


  -> Critical Review: 100%|██████████| 46/46 [00:13<00:00,  3.40it/s]



  - On dataset: OOD Test Data

  - Running baseline evaluation for Gated Attention...


...


  - Running critical review evaluation for Gated Attention...


  -> Critical Review: 100%|██████████| 17/17 [00:09<00:00,  1.70it/s]



--- Analysis Complete ---

Generating PDF report at: /home/takayuki/Desktop/summer2025/plants/experiments/exp-33_ph7_gated_attn_best_ood_loss_Gated Attention.pdf
Generating per-sample analysis pages...


Generating sample pages: 17it [00:11,  1.49it/s]

--- PDF Report Generation Complete ---

Done! You can find your PDF report saved in the experiments folder:
/home/takayuki/Desktop/summer2025/plants/experiments/exp-33_ph7_gated_attn_best_ood_loss_Gated Attention.pdf
